# Complete Model Training Notebook for Dissertation

This notebook trains **ALL models** in the codebase and outputs comprehensive metrics and visualizations.

## Models Covered (25 Total)
1. **Baseline Models (5)**: LSTM, GRU, BiLSTM, Attention LSTM, Transformer
2. **PINN Models (6)**: Baseline PINN, GBM, OU, Black-Scholes, GBM+OU, Global
3. **Advanced PINN (3)**: StackedPINN, ResidualPINN, SpectralPINN
4. **Volatility Models (6)**: Vol-LSTM, Vol-GRU, Vol-Transformer, Vol-PINN, Heston-PINN, Stacked-Vol-PINN
5. **Dual-Phase PDE PINNs (2)**: Burgers PINN, Dual-Phase Burgers PINN
6. **Financial DP-PINNs (3)**: Financial PINN, Financial Dual-Phase PINN, Adaptive Financial Dual-Phase PINN (with GBM, OU, Black-Scholes)

## Setup Instructions
1. Use GPU runtime: Runtime -> Change runtime type -> GPU
2. Clone or upload the repo to Colab
3. Run all cells in order

## Structure
Each model is trained in its own cell with:
- Individual metrics calculation
- Results saved to `{model_name}_results.json`


In [ ]:
#@title 1. Environment Setup
import os
import sys
import subprocess
from pathlib import Path
from getpass import getpass

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', '/content/Dissertaion-Project')

if 'google.colab' in sys.modules:
    REPO_OWNER = 'must1f'
    REPO_NAME = 'Dissertaion-Project'

    if os.path.exists(PROJECT_ROOT) and not Path(PROJECT_ROOT).joinpath('.git').is_dir():
        print(f"Removing incomplete repository at {PROJECT_ROOT}...")
        subprocess.run(['rm', '-rf', PROJECT_ROOT], check=True)

    if not os.path.exists(PROJECT_ROOT):
        github_pat = os.environ.get('GITHUB_PAT')
        if not github_pat:
            print("Please enter your GitHub Personal Access Token (PAT).")
            print("You can generate one here: https://github.com/settings/tokens")
            github_pat = getpass("GitHub PAT: ")
            os.environ['GITHUB_PAT'] = github_pat

        REPO_URL = f'https://{github_pat}@github.com/{REPO_OWNER}/{REPO_NAME}'
        print(f'Cloning private repository to {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
    else:
        print(f'Repository already cloned to {PROJECT_ROOT}. Skipping clone step.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print('GPU Info:')
    print(result.stdout if result.stdout else 'No GPU detected')
except FileNotFoundError:
    print('nvidia-smi not found - running on CPU')

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'matplotlib', 'seaborn',
                'scikit-learn', 'tqdm', 'yfinance', 'python-dotenv', 'pydantic'], check=True)

req_file = Path(PROJECT_ROOT) / 'backend' / 'requirements.txt'
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=False)

print('Setup complete!')

In [ ]:
#@title 2. Import Core Modules
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

try:
    from src.models.model_registry import ModelRegistry
    from src.training.trainer import Trainer
    from src.data.fetcher import DataFetcher
    from src.data.preprocessor import DataPreprocessor
    from src.data.dataset import PhysicsAwareDataset, collate_fn_with_metadata
    from src.utils.config import get_config, get_research_config, get_dynamic_date_range
    from src.utils.reproducibility import set_seed, get_device
    from src.evaluation.metrics import calculate_metrics, calculate_financial_metrics
    from src.evaluation.financial_metrics import compute_strategy_returns, FinancialMetrics
    from src.constants import TRANSACTION_COST, RISK_FREE_RATE, TRADING_DAYS_PER_YEAR
    from src.models.dp_pinn import create_burgers_pinn
    from src.training.dp_pinn_trainer import DPPINNTrainer, TrainingConfig
    from src.utils.sampling import generate_burgers_training_data
    from src.evaluation.pde_evaluator import create_burgers_evaluator
    HAS_SRC = True
    print('Successfully imported src modules - using REAL neural network training')
except ImportError as e:
    HAS_SRC = False
    print(f'Failed to import src modules: {e}')
    raise

try:
    from src.models.financial_dp_pinn import create_financial_dp_pinn
    HAS_FINANCIAL_DP_PINN = True
    print('Financial DP-PINN module imported successfully')
except ImportError as e:
    HAS_FINANCIAL_DP_PINN = False
    print(f'Financial DP-PINN not available: {e}')

try:
    from src.training.volatility_trainer import VolatilityDataPreparer, VolatilityTrainer
    HAS_VOL_TRAINER = True
    print('Volatility trainer imported successfully')
except ImportError as e:
    HAS_VOL_TRAINER = False
    print(f'Volatility trainer not available: {e}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

set_seed(42)
print('Random seed set to 42')

In [ ]:
#@title 2b. Reduce Training Output (Epoch-only Progress)
class _SilentTQDM:
    def __init__(self, iterable, **kwargs):
        self.iterable = iterable
    def __iter__(self):
        for item in self.iterable:
            yield item
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        return False
    def set_postfix(self, *args, **kwargs):
        return None
    def update(self, *args, **kwargs):
        return None

def use_epoch_only_progress():
    import src.training.trainer as trainer_module
    trainer_module.tqdm = _SilentTQDM
    print('Batch-level progress bars disabled (epoch-only progress).')

use_epoch_only_progress()

In [ ]:
#@title 2c. Silence Training Loggers (keep only warnings/errors)
import logging

def _silence_training_loggers():
    for name in [
        'src',
        'src.training',
        'src.training.trainer',
        'src.training.volatility_trainer',
        'src.training.dp_pinn_trainer',
        'src.training.model_checkpointer',
    ]:
        logger = logging.getLogger(name)
        logger.setLevel(logging.WARNING)
        logger.propagate = False
        for handler in logger.handlers:
            handler.setLevel(logging.WARNING)
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.WARNING)
    for handler in root_logger.handlers:
        handler.setLevel(logging.WARNING)
    print('Training loggers set to WARNING (batch-level logs suppressed).')

_silence_training_loggers()

In [ ]:
#@title 3. Define Model Groups (ALL 25 MODELS)
project_root = Path(PROJECT_ROOT)
registry = ModelRegistry(project_root)
research_cfg = get_research_config()

# Baseline Models (5)
BASELINE_MODELS = ['lstm', 'gru', 'bilstm', 'attention_lstm', 'transformer']

# PINN Models (6)
PINN_MODELS = ['baseline_pinn', 'gbm', 'ou', 'black_scholes', 'gbm_ou', 'global']

# Advanced PINN Models (3)
ADVANCED_PINN_MODELS = ['stacked', 'residual', 'spectral_pinn']

# Volatility Models (6)
VOLATILITY_MODELS = ['vol_lstm', 'vol_gru', 'vol_transformer', 'vol_pinn', 'heston_pinn', 'stacked_vol_pinn']

# Dual-Phase PINN (Burgers PDE) Models (2)
DP_MODELS = ['burgers_pinn', 'dual_phase_pinn']

# Financial Dual-Phase PINN Models (3) - GBM, OU, Black-Scholes constraints
FINANCIAL_DP_MODELS = ['financial_pinn', 'financial_dp_pinn', 'adaptive_dual_phase_pinn']

# All price prediction models
PRICE_MODELS = BASELINE_MODELS + PINN_MODELS + ADVANCED_PINN_MODELS + FINANCIAL_DP_MODELS

# Total model count
TOTAL_MODELS = len(PRICE_MODELS) + len(VOLATILITY_MODELS) + len(DP_MODELS)

print(f'ModelRegistry contains {len(registry.models)} model definitions (including aliases)')
print(f'\n=== MODELS TO TRAIN ({TOTAL_MODELS} total) ===')
print(f'\nBaseline models ({len(BASELINE_MODELS)}): {BASELINE_MODELS}')
print(f'PINN models ({len(PINN_MODELS)}): {PINN_MODELS}')
print(f'Advanced PINN ({len(ADVANCED_PINN_MODELS)}): {ADVANCED_PINN_MODELS}')
print(f'Financial DP-PINN models ({len(FINANCIAL_DP_MODELS)}): {FINANCIAL_DP_MODELS}')
print(f'Volatility models ({len(VOLATILITY_MODELS)}): {VOLATILITY_MODELS}')
print(f'DP PINN models ({len(DP_MODELS)}): {DP_MODELS}')
print(f'\nResearch config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')


In [ ]:
#@title 4. Retrieve 10 Years of S&P 500 Data + Prepare Price Prediction Training Data
import yfinance as yf
import time


def download_sp500_10y_data(ticker='^GSPC', max_retries=3, retry_delay=5):
    start_date, end_date = get_dynamic_date_range(10)
    print(f'Downloading 10 years of {ticker} data from Yahoo Finance: {start_date} to {end_date}')

    raw = pd.DataFrame()
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            raw = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                interval='1d',
                progress=False,
                auto_adjust=False,
            )
            if not raw.empty:
                break
            last_error = ValueError(f'No data returned for {ticker} from Yahoo Finance')
        except Exception as e:
            last_error = e

        if attempt < max_retries:
            wait_s = retry_delay * attempt
            print(f'Download attempt {attempt}/{max_retries} failed. Retrying in {wait_s}s...')
            time.sleep(wait_s)

    if raw.empty:
        raise ValueError(f'Yahoo Finance download failed for {ticker} after {max_retries} attempts: {last_error}')

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    raw = raw.reset_index().rename(columns={
        'Date': 'time',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume',
        'Adj Close': 'adjusted_close',
    })

    required_cols = ['time', 'open', 'high', 'low', 'close', 'volume']
    missing_cols = [c for c in required_cols if c not in raw.columns]
    if missing_cols:
        raise ValueError(f'Missing required Yahoo Finance columns: {missing_cols}')

    if 'adjusted_close' not in raw.columns:
        raw['adjusted_close'] = raw['close']

    raw['ticker'] = ticker
    raw['time'] = pd.to_datetime(raw['time']).dt.tz_localize(None)

    df = raw[['time', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'adjusted_close']].copy()
    df = df.dropna(subset=['open', 'high', 'low', 'close']).sort_values('time').reset_index(drop=True)

    print(f'Downloaded {len(df)} rows for {ticker}')
    print(f'Data range: {df["time"].min().date()} to {df["time"].max().date()}')

    return df


sp500_raw_df = download_sp500_10y_data()


def prepare_training_data(raw_price_df, ticker='^GSPC'):
    research_cfg = get_research_config()
    preprocessor = DataPreprocessor()

    if raw_price_df is None or raw_price_df.empty:
        raise ValueError('S&P 500 input dataframe is empty')

    if ticker not in raw_price_df['ticker'].unique():
        raise ValueError(f'{ticker} data missing from downloaded dataset')

    print(f'Preparing features for {ticker} using downloaded Yahoo Finance data (no DB reads)')
    df = preprocessor.process_and_store(raw_price_df.copy())

    research_features = [
        'close', 'volume',
        'log_return', 'simple_return',
        'rolling_volatility_5', 'rolling_volatility_20',
        'momentum_5', 'momentum_20',
        'rsi_14', 'macd', 'macd_signal',
        'bollinger_upper', 'bollinger_lower', 'atr_14',
    ]
    feature_cols = [col for col in research_features if col in df.columns]

    required = ['close', 'log_return']
    missing = [f for f in required if f not in feature_cols]
    if missing:
        raise ValueError(f'Missing required features: {missing}')

    print(f'Using {len(feature_cols)} features')

    train_df, val_df, test_df = preprocessor.split_temporal(df)

    for col in feature_cols:
        train_df[col] = train_df[col].astype(np.float64)
        val_df[col] = val_df[col].astype(np.float64)
        test_df[col] = test_df[col].astype(np.float64)

    train_df_norm, scalers = preprocessor.normalize_features(train_df, feature_cols, method='standard')

    val_df_norm = val_df.copy()
    test_df_norm = test_df.copy()

    for ticker_name in val_df['ticker'].unique():
        if ticker_name in scalers:
            val_mask = val_df_norm['ticker'] == ticker_name
            val_df_norm.loc[val_mask, feature_cols] = scalers[ticker_name].transform(
                val_df_norm.loc[val_mask, feature_cols]
            )

    for ticker_name in test_df['ticker'].unique():
        if ticker_name in scalers:
            test_mask = test_df_norm['ticker'] == ticker_name
            test_df_norm.loc[test_mask, feature_cols] = scalers[ticker_name].transform(
                test_df_norm.loc[test_mask, feature_cols]
            )

    seq_length = research_cfg.sequence_length
    target_col = 'close'

    X_train, y_train, tickers_train = preprocessor.create_sequences(
        train_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )

    X_val, y_val, tickers_val = preprocessor.create_sequences(
        val_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )

    X_test, y_test, tickers_test = preprocessor.create_sequences(
        test_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )

    physics_cols = ['close', 'log_return', 'rolling_volatility_20']
    missing_physics = [c for c in physics_cols if c not in df.columns]
    if missing_physics:
        raise ValueError(f'Missing physics metadata columns: {missing_physics}')

    P_train, _, _ = preprocessor.create_sequences(
        train_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )
    P_val, _, _ = preprocessor.create_sequences(
        val_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )
    P_test, _, _ = preprocessor.create_sequences(
        test_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1
    )

    train_ds = PhysicsAwareDataset(
        X_train, y_train, tickers_train,
        prices=P_train[:, :, 0], returns=P_train[:, :, 1],
        volatilities=P_train[:, :, 2]
    )

    val_ds = PhysicsAwareDataset(
        X_val, y_val, tickers_val,
        prices=P_val[:, :, 0], returns=P_val[:, :, 1],
        volatilities=P_val[:, :, 2]
    )

    test_ds = PhysicsAwareDataset(
        X_test, y_test, tickers_test,
        prices=P_test[:, :, 0], returns=P_test[:, :, 1],
        volatilities=P_test[:, :, 2]
    )

    input_dim = X_train.shape[2]

    print(f'Dataset sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')
    print(f'Input dimension: {input_dim}')
    print(f'Sequence length: {seq_length}')

    return train_ds, val_ds, test_ds, input_dim, scalers


train_ds, val_ds, test_ds, input_dim, scalers = prepare_training_data(sp500_raw_df)


In [ ]:
#@title 5. Prepare Volatility Training Data
def prepare_volatility_data(raw_price_df, ticker='^GSPC'):
    """Prepare data specifically for volatility forecasting models."""
    if not HAS_VOL_TRAINER:
        print('Volatility trainer not available - skipping volatility data prep')
        return None

    if raw_price_df is None or raw_price_df.empty:
        raise ValueError('S&P 500 input dataframe is empty for volatility preparation')

    preprocessor = DataPreprocessor()

    print(f'Preparing volatility data for {ticker} from downloaded Yahoo Finance data (no DB reads)')
    vol_df = preprocessor.process_and_store(raw_price_df.copy())

    single = vol_df[vol_df['ticker'] == ticker].sort_values('time').set_index('time')
    if single.empty:
        raise ValueError(f'No processed rows found for {ticker}')

    preparer = VolatilityDataPreparer(seq_length=40, horizon=5)
    features = preparer.compute_volatility_features(single)

    features = features.dropna()
    aligned_returns = single.loc[features.index, 'log_return']

    dataset = preparer.prepare(
        features.values,
        aligned_returns.values,
        dates=features.index.values
    )

    print(f'Volatility dataset: Train={dataset.X_train.shape[0]}, Val={dataset.X_val.shape[0]}')
    print(f'Volatility features: {dataset.X_train.shape[2]}')

    return dataset


vol_dataset = prepare_volatility_data(sp500_raw_df)


In [ ]:
#@title 6. Training Helper Functions and Data Loaders
train_loader = DataLoader(train_ds, batch_size=research_cfg.batch_size,
                          shuffle=True, collate_fn=collate_fn_with_metadata)
val_loader = DataLoader(val_ds, batch_size=research_cfg.batch_size,
                        shuffle=False, collate_fn=collate_fn_with_metadata)
test_loader = DataLoader(test_ds, batch_size=research_cfg.batch_size,
                         shuffle=False, collate_fn=collate_fn_with_metadata)

results_dir = project_root / 'results' / 'colab_runs'
results_dir.mkdir(parents=True, exist_ok=True)

# Storage for all results
all_results = {}
baseline_results = {}
pinn_results = {}
advanced_results = {}
volatility_results = {}
dp_results = {}
financial_dp_results = {}


def _to_python(obj):
    if isinstance(obj, dict):
        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_python(v) for v in obj]
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def _extract_close_stats(scalers):
    if not scalers or not isinstance(scalers, dict):
        return None, None
    if len(scalers) != 1:
        return None, None

    sc = next(iter(scalers.values()))
    mean = getattr(sc, 'mean_', None)
    std = getattr(sc, 'scale_', None)
    if mean is None or std is None:
        return None, None

    mean_val = float(np.squeeze(mean[0]) if hasattr(mean, 'shape') and len(mean.shape) > 0 else float(mean))
    std_val = float(np.squeeze(std[0]) if hasattr(std, 'shape') and len(std.shape) > 0 else float(std))
    return mean_val, std_val


def compute_all_evaluation_metrics(predictions, targets, model_name, scalers=None):
    metrics = {}

    price_mean, price_std = _extract_close_stats(scalers)

    pred_metrics = calculate_metrics(
        targets,
        predictions,
        prefix='',
        price_mean=price_mean,
        price_std=price_std,
    )

    metrics.update({
        'rmse': pred_metrics.get('rmse'),
        'mae': pred_metrics.get('mae'),
        'mape': pred_metrics.get('mape'),
        'mse': pred_metrics.get('mse'),
        'r2': pred_metrics.get('r2'),
        'directional_accuracy': pred_metrics.get('directional_accuracy'),
    })

    try:
        strategy_returns = compute_strategy_returns(
            predictions=predictions,
            actual_prices=targets,
            are_returns=False,
            transaction_cost=TRANSACTION_COST,
            price_mean=price_mean,
            price_std=price_std,
        )

        fin_metrics = calculate_financial_metrics(
            returns=strategy_returns,
            risk_free_rate=RISK_FREE_RATE,
            periods_per_year=TRADING_DAYS_PER_YEAR,
            prefix='',
        )

        metrics.update({
            'sharpe_ratio': fin_metrics.get('sharpe_ratio'),
            'sortino_ratio': fin_metrics.get('sortino_ratio'),
            'max_drawdown': fin_metrics.get('max_drawdown'),
            'calmar_ratio': fin_metrics.get('calmar_ratio'),
            'total_return': fin_metrics.get('total_return'),
            'volatility': fin_metrics.get('volatility'),
            'win_rate': fin_metrics.get('win_rate'),
        })

        if len(strategy_returns) > 0:
            metrics['annualized_return'] = FinancialMetrics.annualized_return(
                strategy_returns, periods_per_year=TRADING_DAYS_PER_YEAR
            ) * 100

    except Exception as e:
        print(f'Warning: Financial metrics failed for {model_name}: {e}')

    return metrics


def evaluate_model(model, loader, model_name, scalers=None):
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in loader:
            sequences, target, metadata = batch
            sequences = sequences.to(device)
            target = target.to(device)

            output = model(sequences)
            if isinstance(output, tuple):
                output = output[0]

            predictions.append(np.asarray(output.cpu().numpy()).reshape(-1))
            targets.append(np.asarray(target.cpu().numpy()).reshape(-1))

    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)

    metrics = compute_all_evaluation_metrics(
        predictions=predictions,
        targets=targets,
        model_name=model_name,
        scalers=scalers,
    )

    return predictions, targets, metrics


def train_price_model(model_key, epochs=None, scalers=None, enable_physics=None):
    print(f'\n{"="*60}')
    print(f'Training: {model_key}')
    print(f'{"="*60}')

    epochs = epochs or research_cfg.epochs

    model = registry.create_model(
        model_type=model_key,
        input_dim=input_dim,
        hidden_dim=research_cfg.hidden_dim,
        num_layers=research_cfg.num_layers,
        dropout=research_cfg.dropout,
    )

    if model is None:
        print(f'Failed to create model: {model_key}')
        return None

    is_pinn = hasattr(model, 'compute_loss')
    if enable_physics is None:
        enable_physics = is_pinn

    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'Physics-enabled training: {bool(enable_physics and is_pinn)}')

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        research_mode=True,
    )

    history = {
        'epochs': [],
        'train_loss': [],
        'val_loss': [],
        'data_loss': [],
        'physics_loss': [],
        'gbm_loss': [],
        'ou_loss': [],
        'bs_loss': [],
        'learning_rates': [],
    }

    best_val_loss = float('inf')

    pbar = tqdm(range(1, epochs + 1), desc=model_key)
    for epoch in pbar:
        train_loss, train_metrics = trainer.train_epoch(enable_physics=bool(enable_physics and is_pinn))
        val_loss, _ = trainer.validate_epoch(enable_physics=bool(enable_physics and is_pinn))

        history['epochs'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['data_loss'].append(train_metrics.get('train_data_loss'))
        history['physics_loss'].append(train_metrics.get('train_physics_loss'))
        history['gbm_loss'].append(train_metrics.get('gbm_loss'))
        history['ou_loss'].append(train_metrics.get('ou_loss'))
        history['bs_loss'].append(train_metrics.get('bs_loss'))
        history['learning_rates'].append(trainer.optimizer.param_groups[0]['lr'])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trainer.save_checkpoint(epoch=epoch, val_loss=val_loss,
                                    is_best=True, model_name=model_key)

        pbar.set_postfix({'train': f'{train_loss:.4f}', 'val': f'{val_loss:.4f}'})

    predictions, targets, metrics = evaluate_model(
        trainer.model,
        test_loader,
        model_key,
        scalers=scalers,
    )

    print(f'\n--- {model_key} Test Metrics ---')
    print(f'  RMSE: {metrics.get("rmse", float("nan")):.4f}')
    print(f'  MAE: {metrics.get("mae", float("nan")):.4f}')
    print(f'  R²: {metrics.get("r2", float("nan")):.4f}')
    print(f'  Directional Accuracy: {metrics.get("directional_accuracy", float("nan")):.2f}%')
    print(f'  Sharpe Ratio: {metrics.get("sharpe_ratio", float("nan")):.3f}')
    print(f'  Sortino Ratio: {metrics.get("sortino_ratio", float("nan")):.3f}')
    print(f'  Max Drawdown: {metrics.get("max_drawdown", float("nan")):.2f}')

    payload = {
        'model': model_key,
        'history': history,
        'test_metrics': metrics,
        'metrics': metrics,
        'training_completed': True,
        'best_val_loss': best_val_loss,
        'epochs_trained': epochs,
    }

    # Save to individual file
    output_file = results_dir / f'{model_key}_results.json'
    with open(output_file, 'w') as f:
        json.dump(_to_python(payload), f, indent=2)
    print(f'Results saved to: {output_file}')

    return {
        'history': history,
        'metrics': metrics,
        'predictions': predictions,
        'targets': targets,
    }


def train_volatility_model(model_key, dataset, epochs=30):
    if dataset is None or not HAS_VOL_TRAINER:
        print(f'Skipping {model_key} - volatility trainer not available')
        return None

    print(f'\n{"="*60}')
    print(f'Training Volatility Model: {model_key}')
    print(f'{"="*60}')

    model = registry.create_model(
        model_key,
        input_dim=dataset.X_train.shape[2],
        hidden_dim=64,
        num_layers=2,
        dropout=0.1,
    )

    if model is None:
        print(f'Failed to create model: {model_key}')
        return None

    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

    trainer = VolatilityTrainer(
        model,
        learning_rate=1e-3,
        batch_size=64,
        max_epochs=epochs,
        patience=10,
        device=device,
        checkpoint_dir=results_dir,
    )

    enable_physics = 'pinn' in model_key or 'heston' in model_key
    history = trainer.fit(dataset, enable_physics=enable_physics, verbose=True)

    val_losses = history.get('val_loss', [])
    best_epoch = int(np.argmin(val_losses)) if val_losses else None

    metrics = {
        'final_train_loss': history.get('train_loss', [np.nan])[-1] if history.get('train_loss') else np.nan,
        'final_val_loss': history.get('val_loss', [np.nan])[-1] if history.get('val_loss') else np.nan,
        'best_val_loss': np.min(history.get('val_loss', [np.nan])) if history.get('val_loss') else np.nan,
        'final_val_mse': history.get('val_mse', [np.nan])[-1] if history.get('val_mse') else np.nan,
        'final_val_mae': history.get('val_mae', [np.nan])[-1] if history.get('val_mae') else np.nan,
        'final_val_qlike': history.get('val_qlike', [np.nan])[-1] if history.get('val_qlike') else np.nan,
        'final_val_r2': history.get('val_r2', [np.nan])[-1] if history.get('val_r2') else np.nan,
        'epochs_trained': len(history.get('train_loss', [])),
    }

    if best_epoch is not None:
        metrics.update({
            'best_epoch': best_epoch + 1,
            'best_val_mse': history.get('val_mse', [np.nan])[best_epoch] if history.get('val_mse') else np.nan,
            'best_val_mae': history.get('val_mae', [np.nan])[best_epoch] if history.get('val_mae') else np.nan,
            'best_val_qlike': history.get('val_qlike', [np.nan])[best_epoch] if history.get('val_qlike') else np.nan,
            'best_val_r2': history.get('val_r2', [np.nan])[best_epoch] if history.get('val_r2') else np.nan,
        })

    if dataset.X_test is not None and dataset.y_test is not None:
        try:
            test_metrics = trainer.evaluate(dataset.X_test, dataset.y_test)
            for key, value in test_metrics.items():
                if isinstance(value, (int, float, np.integer, np.floating)):
                    metrics[f'test_{key}'] = float(value)
        except Exception as e:
            print(f'Warning: Test metrics failed for {model_key}: {e}')

    print(f'\n--- {model_key} Volatility Metrics ---')
    print(f'  Best Val Loss: {metrics.get("best_val_loss", float("nan")):.6f}')
    print(f'  Best Val QLIKE: {metrics.get("best_val_qlike", float("nan")):.6f}')
    print(f'  Best Val R²: {metrics.get("best_val_r2", float("nan")):.4f}')

    payload = {
        'model': model_key,
        'history': history,
        'metrics': metrics,
        'training_completed': True,
    }

    # Save to individual file
    output_file = results_dir / f'{model_key}_results.json'
    with open(output_file, 'w') as f:
        json.dump(_to_python(payload), f, indent=2)
    print(f'Results saved to: {output_file}')

    return {
        'history': history,
        'metrics': metrics,
    }


def train_financial_dp_model(model_key, epochs=50):
    """Train Financial DP-PINN models using price prediction data."""
    if not HAS_FINANCIAL_DP_PINN:
        print(f'Skipping {model_key} - Financial DP-PINN module not available')
        return None

    print(f'\n{"="*60}')
    print(f'Training Financial DP-PINN Model: {model_key}')
    print(f'{"="*60}')

    # Create model from registry (supports financial_pinn, financial_dp_pinn, adaptive_dual_phase_pinn)
    model = registry.create_model(
        model_type=model_key,
        input_dim=input_dim,
        hidden_dim=research_cfg.hidden_dim,
        num_layers=research_cfg.num_layers,
        dropout=research_cfg.dropout,
    )

    # Backward-compatible fallback for legacy keys
    if model is None and model_key in ('financial_pinn', 'financial_dp_pinn'):
        model_type = 'standard' if model_key == 'financial_pinn' else 'dual_phase'
        model = create_financial_dp_pinn(
            model_type=model_type,
            input_dim=input_dim,
            hidden_dim=research_cfg.hidden_dim,
            num_layers=research_cfg.num_layers,
            dropout=research_cfg.dropout,
        )

    if model is None:
        print(f'Failed to create model: {model_key}')
        return None

    model = model.to(device)
    is_dual_phase = hasattr(model, 'phase1_net') or hasattr(model, 'phase1_network')

    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'Dual-Phase: {is_dual_phase}')
    print(f'Physics Constraints: GBM, OU, Black-Scholes')

    # Use standard trainer with physics enabled
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        research_mode=True,
    )

    history = {
        'epochs': [],
        'train_loss': [],
        'val_loss': [],
        'data_loss': [],
        'physics_loss': [],
        'gbm_loss': [],
        'ou_loss': [],
        'bs_loss': [],
    }

    best_val_loss = float('inf')

    pbar = tqdm(range(1, epochs + 1), desc=model_key)
    for epoch in pbar:
        train_loss, train_metrics = trainer.train_epoch(enable_physics=True)
        val_loss, _ = trainer.validate_epoch(enable_physics=True)

        history['epochs'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['data_loss'].append(train_metrics.get('train_data_loss'))
        history['physics_loss'].append(train_metrics.get('train_physics_loss'))
        history['gbm_loss'].append(train_metrics.get('gbm_loss'))
        history['ou_loss'].append(train_metrics.get('ou_loss'))
        history['bs_loss'].append(train_metrics.get('bs_loss'))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trainer.save_checkpoint(epoch=epoch, val_loss=val_loss,
                                    is_best=True, model_name=model_key)

        pbar.set_postfix({'train': f'{train_loss:.4f}', 'val': f'{val_loss:.4f}'})

    # Evaluate
    predictions, targets, metrics = evaluate_model(
        trainer.model,
        test_loader,
        model_key,
        scalers=scalers,
    )

    # Add physics info
    metrics['has_gbm_constraint'] = True
    metrics['has_ou_constraint'] = True
    metrics['has_bs_constraint'] = True
    metrics['is_dual_phase'] = is_dual_phase

    print(f'\n--- {model_key} Test Metrics ---')
    print(f'  RMSE: {metrics.get("rmse", float("nan")):.4f}')
    print(f'  MAE: {metrics.get("mae", float("nan")):.4f}')
    print(f'  R²: {metrics.get("r2", float("nan")):.4f}')
    print(f'  Directional Accuracy: {metrics.get("directional_accuracy", float("nan")):.2f}%')
    print(f'  Sharpe Ratio: {metrics.get("sharpe_ratio", float("nan")):.3f}')

    payload = {
        'model': model_key,
        'history': history,
        'test_metrics': metrics,
        'metrics': metrics,
        'training_completed': True,
        'best_val_loss': best_val_loss,
        'epochs_trained': epochs,
        'physics_constraints': ['GBM', 'OU', 'Black-Scholes'],
    }

    # Save to individual file
    output_file = results_dir / f'{model_key}_results.json'
    with open(output_file, 'w') as f:
        json.dump(_to_python(payload), f, indent=2)
    print(f'Results saved to: {output_file}')

    return {
        'history': history,
        'metrics': metrics,
        'predictions': predictions,
        'targets': targets,
    }

print('Training functions defined!')
print(f'Results will be saved to: {results_dir}')


---
# Baseline Models (5 Models)
Each model trained separately with individual metrics and file output.

In [ ]:
#@title 7a. Train LSTM Model
try:
    result = train_price_model('lstm', scalers=scalers, enable_physics=False)
    if result:
        baseline_results['lstm'] = result
        all_results['lstm'] = result
except Exception as e:
    print(f'Error training lstm: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 7b. Train GRU Model
try:
    result = train_price_model('gru', scalers=scalers, enable_physics=False)
    if result:
        baseline_results['gru'] = result
        all_results['gru'] = result
except Exception as e:
    print(f'Error training gru: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 7c. Train BiLSTM Model
try:
    result = train_price_model('bilstm', scalers=scalers, enable_physics=False)
    if result:
        baseline_results['bilstm'] = result
        all_results['bilstm'] = result
except Exception as e:
    print(f'Error training bilstm: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 7d. Train Attention LSTM Model
try:
    result = train_price_model('attention_lstm', scalers=scalers, enable_physics=False)
    if result:
        baseline_results['attention_lstm'] = result
        all_results['attention_lstm'] = result
except Exception as e:
    print(f'Error training attention_lstm: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 7e. Train Transformer Model
try:
    result = train_price_model('transformer', scalers=scalers, enable_physics=False)
    if result:
        baseline_results['transformer'] = result
        all_results['transformer'] = result
except Exception as e:
    print(f'Error training transformer: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== Baseline Models Complete: {len(baseline_results)}/{len(BASELINE_MODELS)} ===')

---
# PINN Models (6 Models)
Physics-Informed Neural Networks with various constraints.

In [ ]:
#@title 8a. Train Baseline PINN Model
try:
    result = train_price_model('baseline_pinn', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['baseline_pinn'] = result
        all_results['baseline_pinn'] = result
except Exception as e:
    print(f'Error training baseline_pinn: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 8b. Train GBM PINN Model
try:
    result = train_price_model('gbm', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['gbm'] = result
        all_results['gbm'] = result
except Exception as e:
    print(f'Error training gbm: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 8c. Train OU PINN Model
try:
    result = train_price_model('ou', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['ou'] = result
        all_results['ou'] = result
except Exception as e:
    print(f'Error training ou: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 8d. Train Black-Scholes PINN Model
try:
    result = train_price_model('black_scholes', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['black_scholes'] = result
        all_results['black_scholes'] = result
except Exception as e:
    print(f'Error training black_scholes: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 8e. Train GBM+OU PINN Model
try:
    result = train_price_model('gbm_ou', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['gbm_ou'] = result
        all_results['gbm_ou'] = result
except Exception as e:
    print(f'Error training gbm_ou: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 8f. Train Global PINN Model
try:
    result = train_price_model('global', scalers=scalers, enable_physics=True)
    if result:
        pinn_results['global'] = result
        all_results['global'] = result
except Exception as e:
    print(f'Error training global: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== PINN Models Complete: {len(pinn_results)}/{len(PINN_MODELS)} ===')

---
# Advanced PINN Models (3 Models)
Stacked, Residual, and Spectral architectures.

In [ ]:
#@title 9a. Train Stacked PINN Model
try:
    result = train_price_model('stacked', scalers=scalers, enable_physics=True)
    if result:
        advanced_results['stacked'] = result
        all_results['stacked'] = result
except Exception as e:
    print(f'Error training stacked: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 9b. Train Residual PINN Model
try:
    result = train_price_model('residual', scalers=scalers, enable_physics=True)
    if result:
        advanced_results['residual'] = result
        all_results['residual'] = result
except Exception as e:
    print(f'Error training residual: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 9c. Train Spectral PINN Model
try:
    result = train_price_model('spectral_pinn', scalers=scalers, enable_physics=True)
    if result:
        advanced_results['spectral_pinn'] = result
        all_results['spectral_pinn'] = result
except Exception as e:
    print(f'Error training spectral_pinn: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== Advanced PINN Models Complete: {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)} ===')

---
# Volatility Models (6 Models)
Volatility forecasting models including Heston-PINN.

In [ ]:
#@title 10a. Train Vol-LSTM Model
try:
    result = train_volatility_model('vol_lstm', vol_dataset, epochs=30)
    if result:
        volatility_results['vol_lstm'] = result
        all_results['vol_lstm'] = result
except Exception as e:
    print(f'Error training vol_lstm: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 10b. Train Vol-GRU Model
try:
    result = train_volatility_model('vol_gru', vol_dataset, epochs=30)
    if result:
        volatility_results['vol_gru'] = result
        all_results['vol_gru'] = result
except Exception as e:
    print(f'Error training vol_gru: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 10c. Train Vol-Transformer Model
try:
    result = train_volatility_model('vol_transformer', vol_dataset, epochs=30)
    if result:
        volatility_results['vol_transformer'] = result
        all_results['vol_transformer'] = result
except Exception as e:
    print(f'Error training vol_transformer: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 10d. Train Vol-PINN Model
try:
    result = train_volatility_model('vol_pinn', vol_dataset, epochs=30)
    if result:
        volatility_results['vol_pinn'] = result
        all_results['vol_pinn'] = result
except Exception as e:
    print(f'Error training vol_pinn: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 10e. Train Heston-PINN Model
try:
    result = train_volatility_model('heston_pinn', vol_dataset, epochs=30)
    if result:
        volatility_results['heston_pinn'] = result
        all_results['heston_pinn'] = result
except Exception as e:
    print(f'Error training heston_pinn: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 10f. Train Stacked-Vol-PINN Model
try:
    result = train_volatility_model('stacked_vol_pinn', vol_dataset, epochs=30)
    if result:
        volatility_results['stacked_vol_pinn'] = result
        all_results['stacked_vol_pinn'] = result
except Exception as e:
    print(f'Error training stacked_vol_pinn: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== Volatility Models Complete: {len(volatility_results)}/{len(VOLATILITY_MODELS)} ===')

---
# Dual-Phase PINN Models (2 Models)
Burgers PDE-based physics-informed models.

In [ ]:
#@title 11a. Train Burgers PINN Model
dp_checkpoints_dir = results_dir / 'dp_pinn'
dp_checkpoints_dir.mkdir(parents=True, exist_ok=True)

dp_config = TrainingConfig(device=str(device))
burgers_data = generate_burgers_training_data(device=device)
burgers_evaluator = create_burgers_evaluator(device=device)

try:
    trainer_burg = DPPINNTrainer(create_burgers_pinn(model_type='standard').to(device), dp_config, device=device)
    trainer_burg.train_standard_pinn(burgers_data, eval_fn=burgers_evaluator.relative_l2_error)
    trainer_burg.save_checkpoint(dp_checkpoints_dir / 'burgers_pinn.ckpt', extra_info={'model_key': 'burgers_pinn'})
    burg_metrics = burgers_evaluator.evaluate_all(trainer_burg.model)
    
    dp_results['burgers_pinn'] = {
        'metrics': burg_metrics.to_dict(),
        'history': trainer_burg.history.to_dict(),
    }
    all_results['burgers_pinn'] = dp_results['burgers_pinn']
    
    # Save to individual file
    output_file = results_dir / 'burgers_pinn_results.json'
    with open(output_file, 'w') as f:
        json.dump(_to_python(dp_results['burgers_pinn']), f, indent=2)
    print(f'Results saved to: {output_file}')
    
    print(f"\n--- burgers_pinn Metrics ---")
    print(f"  L2_rel: {burg_metrics.to_dict().get('relative_l2_error', float('nan')):.6f}")
    print(f"  Max Error: {burg_metrics.to_dict().get('max_error', float('nan')):.6f}")
except Exception as e:
    print(f'Error training burgers_pinn: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 11b. Train Dual-Phase PINN Model
try:
    trainer_dp = DPPINNTrainer(create_burgers_pinn(model_type='dual_phase').to(device), dp_config, device=device)
    phase1_history = trainer_dp.train_phase1(burgers_data, eval_fn=burgers_evaluator.relative_l2_error)
    phase1_history_dict = phase1_history.to_dict()
    phase2_history = trainer_dp.train_phase2(burgers_data, eval_fn=burgers_evaluator.relative_l2_error)
    phase2_history_dict = phase2_history.to_dict()
    trainer_dp.save_checkpoint(dp_checkpoints_dir / 'dual_phase_pinn.ckpt', extra_info={'model_key': 'dual_phase_pinn'})
    dp_metrics = burgers_evaluator.evaluate_all(trainer_dp.model)
    
    dp_results['dual_phase_pinn'] = {
        'metrics': dp_metrics.to_dict(),
        'phase1_history': phase1_history_dict,
        'phase2_history': phase2_history_dict,
    }
    all_results['dual_phase_pinn'] = dp_results['dual_phase_pinn']
    
    # Save to individual file
    output_file = results_dir / 'dual_phase_pinn_results.json'
    with open(output_file, 'w') as f:
        json.dump(_to_python(dp_results['dual_phase_pinn']), f, indent=2)
    print(f'Results saved to: {output_file}')
    
    print(f"\n--- dual_phase_pinn Metrics ---")
    print(f"  L2_rel: {dp_metrics.to_dict().get('relative_l2_error', float('nan')):.6f}")
    print(f"  Max Error: {dp_metrics.to_dict().get('max_error', float('nan')):.6f}")
except Exception as e:
    print(f'Error training dual_phase_pinn: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== DP PINN Models Complete: {len(dp_results)}/{len(DP_MODELS)} ===')

---
# Financial Dual-Phase PINN Models (3 Models)
Physics-informed models with GBM, OU, and Black-Scholes financial constraints.


In [ ]:
#@title 11c. Train Financial PINN Model
try:
    result = train_financial_dp_model('financial_pinn', epochs=research_cfg.epochs)
    if result:
        financial_dp_results['financial_pinn'] = result
        all_results['financial_pinn'] = result
except Exception as e:
    print(f'Error training financial_pinn: {e}')
    import traceback
    traceback.print_exc()

In [ ]:
#@title 11d. Train Financial Dual-Phase PINN Model
try:
    result = train_financial_dp_model('financial_dp_pinn', epochs=research_cfg.epochs)
    if result:
        financial_dp_results['financial_dp_pinn'] = result
        all_results['financial_dp_pinn'] = result
except Exception as e:
    print(f'Error training financial_dp_pinn: {e}')
    import traceback
    traceback.print_exc()

print(f'\n=== Financial DP-PINN Models Complete: {len(financial_dp_results)}/{len(FINANCIAL_DP_MODELS)} ===')

---
# Adaptive Financial Dual-Phase PINN (1 Model)
Volatility-gated dual-phase financial PINN with OU-first physics, log-space GBM, and continuity penalty.

In [ ]:
#@title 11e. Train Adaptive Financial Dual-Phase PINN Model
try:
    result = train_financial_dp_model('adaptive_dual_phase_pinn', epochs=research_cfg.epochs)
    if result:
        financial_dp_results['adaptive_dual_phase_pinn'] = result
        all_results['adaptive_dual_phase_pinn'] = result
except Exception as e:
    print(f'Error training adaptive_dual_phase_pinn: {e}')
    import traceback
    traceback.print_exc()


print(f'\n=== Adaptive Financial DP-PINN Complete: {len(financial_dp_results)}/{len(FINANCIAL_DP_MODELS)} ===')


---
# Summary and Visualization

In [ ]:
#@title 12. Combine All Results and Summary
price_results = {**baseline_results, **pinn_results, **advanced_results, **financial_dp_results}

print(f'=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}')
print(f'  - PINN: {len(pinn_results)}')
print(f'  - Advanced PINN: {len(advanced_results)}')
print(f'  - Financial DP-PINN: {len(financial_dp_results)}')
print(f'  - Volatility: {len(volatility_results)}')
print(f'  - DP PINN: {len(dp_results)}')
print(f'\nAll models: {list(all_results.keys())}')

In [ ]:
#@title 13. Metrics Coverage Audit (All Models)
def audit_metrics():
    audit = {'price': {}, 'volatility': {}, 'dp': {}}
    required_price = ['rmse', 'mae', 'mape', 'r2', 'directional_accuracy', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown']
    required_vol = ['best_val_loss', 'best_val_qlike', 'best_val_r2', 'final_val_mse', 'final_val_mae']
    required_dp = ['relative_l2_error', 'max_error', 'mean_error', 'mean_residual']
    
    for name, res in price_results.items():
        metrics = res.get('metrics', {})
        missing = [m for m in required_price if metrics.get(m) is None]
        audit['price'][name] = {'missing': missing, 'ok': len(missing) == 0}
    
    for name, res in volatility_results.items():
        metrics = res.get('metrics', {})
        missing = [m for m in required_vol if metrics.get(m) is None]
        audit['volatility'][name] = {'missing': missing, 'ok': len(missing) == 0}
    
    for name, res in dp_results.items():
        metrics = res.get('metrics', {})
        missing = [m for m in required_dp if metrics.get(m) is None]
        audit['dp'][name] = {'missing': missing, 'ok': len(missing) == 0}
    
    audit['artifacts'] = {
        'price_summary_csv': (results_dir / 'price_model_summary.csv').exists(),
        'vol_summary_csv': (results_dir / 'volatility_model_summary.csv').exists(),
        'dp_summary_csv': (results_dir / 'dp_pinn_summary.csv').exists(),
        'price_training_png': (results_dir / 'price_training_curves.png').exists(),
        'vol_training_png': (results_dir / 'volatility_training_curves.png').exists(),
        'metrics_comparison_png': (results_dir / 'metrics_comparison.png').exists(),
        'vol_metrics_png': (results_dir / 'volatility_metrics_comparison.png').exists(),
        'dp_training_png': (results_dir / 'dp_pinn_training_curves.png').exists(),
        'pred_vs_actual_png': (results_dir / 'predictions_vs_actual_all_price_models.png').exists(),
    }
    
    with open(results_dir / 'metrics_audit.json', 'w') as f:
        json.dump(_to_python(audit), f, indent=2)
    
    print('Metrics audit saved to', results_dir / 'metrics_audit.json')
    for category, models in audit.items():
        if category == 'artifacts':
            continue
        for name, entry in models.items():
            status = 'OK' if entry['ok'] else f"Missing: {entry['missing']}"
            print(f"{category.upper()} - {name}: {status}")
    return audit

audit_results = audit_metrics()

In [ ]:
#@title 14. Generate Price Models Metrics Summary
summary_rows = []

for model_key, result in price_results.items():
    metrics = result['metrics']
    
    if model_key in BASELINE_MODELS:
        model_type = 'Baseline'
    elif model_key in PINN_MODELS:
        model_type = 'PINN'
    elif model_key in FINANCIAL_DP_MODELS:
        model_type = 'Financial DP-PINN'
    else:
        model_type = 'Advanced PINN'
    
    summary_rows.append({
        'Model': model_key,
        'Type': model_type,
        'RMSE': metrics.get('rmse'),
        'MAE': metrics.get('mae'),
        'MAPE': metrics.get('mape'),
        'R2': metrics.get('r2'),
        'Dir. Acc. (%)': metrics.get('directional_accuracy'),
        'Sharpe': metrics.get('sharpe_ratio'),
        'Sortino': metrics.get('sortino_ratio'),
        'Max DD (%)': metrics.get('max_drawdown', 0) * 100 if metrics.get('max_drawdown') else None
    })

price_summary_df = pd.DataFrame(summary_rows)
if not price_summary_df.empty and 'RMSE' in price_summary_df.columns:
    price_summary_df = price_summary_df.sort_values('RMSE')

print('\n' + '='*80)
print('PRICE PREDICTION MODEL PERFORMANCE SUMMARY')
print('='*80)
display(price_summary_df.round(4))

price_summary_df.to_csv(results_dir / 'price_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "price_model_summary.csv"}')

In [ ]:
#@title 15. Generate Volatility Models Metrics Summary
vol_summary_rows = []

for model_key, result in volatility_results.items():
    metrics = result['metrics']

    is_physics = 'pinn' in model_key or 'heston' in model_key

    vol_summary_rows.append({
        'Model': model_key,
        'Type': 'Physics-Informed' if is_physics else 'Baseline',
        'Best Val Loss': metrics.get('best_val_loss'),
        'Best Val QLIKE': metrics.get('best_val_qlike'),
        'Best Val R2': metrics.get('best_val_r2'),
        'Final Val MSE': metrics.get('final_val_mse'),
        'Final Val MAE': metrics.get('final_val_mae'),
        'Epochs Trained': metrics.get('epochs_trained'),
    })

vol_summary_df = pd.DataFrame(vol_summary_rows)

if not vol_summary_df.empty:
    sort_col = 'Best Val QLIKE' if 'Best Val QLIKE' in vol_summary_df.columns else 'Best Val Loss'
    vol_summary_df = vol_summary_df.sort_values(sort_col)

print('\n' + '='*80)
print('VOLATILITY MODEL PERFORMANCE SUMMARY')
print('='*80)
display(vol_summary_df.round(6))

vol_summary_df.to_csv(results_dir / 'volatility_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "volatility_model_summary.csv"}')

In [ ]:
#@title 16. Generate DP PINN Metrics Summary
if dp_results:
    dp_summary_rows = []
    for model_key, res in dp_results.items():
        m = res['metrics']
        dp_summary_rows.append({
            'Model': model_key,
            'Relative L2': m.get('relative_l2_error'),
            'Max Error': m.get('max_error'),
            'Mean Error': m.get('mean_error'),
            'Mean Residual': m.get('mean_residual'),
        })

    dp_summary_df = pd.DataFrame(dp_summary_rows)
    print('\n' + '='*80)
    print('DP PINN MODEL PERFORMANCE SUMMARY')
    print('='*80)
    display(dp_summary_df.round(6))
    dp_summary_df.to_csv(results_dir / 'dp_pinn_summary.csv', index=False)
    print(f"DP PINN summary saved to {results_dir / 'dp_pinn_summary.csv'}")
else:
    print('No DP PINN results available')

In [ ]:
#@title 17. Plot Training Curves - Price Models
if price_results:
    n_models = len(price_results)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = np.array(axes).reshape(-1)

    for idx, (model_key, result) in enumerate(price_results.items()):
        ax = axes[idx]
        history = result['history']
        
        ax.plot(history['epochs'], history['train_loss'], label='Train', linewidth=2)
        ax.plot(history['epochs'], history['val_loss'], label='Val', linewidth=2, linestyle='--')
        
        ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)

    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Training Curves - Price Prediction Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'price_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Training curves saved to {results_dir / "price_training_curves.png"}')
else:
    print('No price models trained')

In [ ]:
#@title 18. Plot Training Curves - Volatility Models
if volatility_results:
    n_vol = len(volatility_results)
    n_cols_vol = min(3, n_vol)
    n_rows_vol = (n_vol + n_cols_vol - 1) // n_cols_vol
    
    fig, axes = plt.subplots(n_rows_vol, n_cols_vol, figsize=(15, 4*n_rows_vol))
    if n_vol > 1:
        axes = axes.flatten()
    else:
        axes = [axes]
    
    for idx, (model_key, result) in enumerate(volatility_results.items()):
        ax = axes[idx]
        history = result['history']
        
        if 'train_loss' in history and 'val_loss' in history:
            epochs = range(1, len(history['train_loss']) + 1)
            ax.plot(epochs, history['train_loss'], label='Train', linewidth=2)
            ax.plot(epochs, history['val_loss'], label='Val', linewidth=2, linestyle='--')
        
        ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    for idx in range(n_vol, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Training Curves - Volatility Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'volatility_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Volatility training curves saved to {results_dir / "volatility_training_curves.png"}')
else:
    print('No volatility models trained')

In [ ]:
#@title 19. Plot DP PINN Training Curves
if dp_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Standard Burgers PINN losses
    std_hist = dp_results.get('burgers_pinn', {}).get('history', {})
    if std_hist:
        axes[0].plot(std_hist.get('iterations', []), std_hist.get('losses', []), label='Total Loss')
        axes[0].plot(std_hist.get('iterations', []), std_hist.get('pde_losses', []), label='PDE Loss', alpha=0.8)
        axes[0].set_title('Burgers PINN Loss')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylabel('Loss')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

    # Dual-phase losses
    dp_hist = dp_results.get('dual_phase_pinn', {}).get('phase2_history', {})
    if dp_hist:
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('losses', []), label='Total Loss')
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('pde_losses', []), label='PDE Loss', alpha=0.8)
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('intermediate_losses', []), label='Intermediate', alpha=0.8)
        axes[1].set_title('Dual-Phase PINN Loss (Phase 2)')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Loss')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(results_dir / 'dp_pinn_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'DP PINN training curves saved to {results_dir / "dp_pinn_training_curves.png"}')
else:
    print('No DP PINN results available')

In [ ]:
#@title 20. Plot Metrics Comparison - All Price Models
if len(price_summary_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    colors = []
    for model in price_summary_df['Model']:
        if model in BASELINE_MODELS:
            colors.append('#2E86AB')
        elif model in PINN_MODELS:
            colors.append('#A23B72')
        else:
            colors.append('#F18F01')

    ax1 = axes[0, 0]
    ax1.barh(price_summary_df['Model'], price_summary_df['RMSE'], color=colors)
    ax1.set_xlabel('RMSE (lower is better)')
    ax1.set_title('RMSE by Model')
    ax1.invert_yaxis()

    ax2 = axes[0, 1]
    ax2.barh(price_summary_df['Model'], price_summary_df['Dir. Acc. (%)'], color=colors)
    ax2.set_xlabel('Directional Accuracy %')
    ax2.set_title('Directional Accuracy by Model')
    ax2.invert_yaxis()
    ax2.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='Random (50%)')

    ax3 = axes[1, 0]
    sharpe_sorted = price_summary_df.sort_values('Sharpe', ascending=True)
    colors_sharpe = []
    for model in sharpe_sorted['Model']:
        if model in BASELINE_MODELS:
            colors_sharpe.append('#2E86AB')
        elif model in PINN_MODELS:
            colors_sharpe.append('#A23B72')
        else:
            colors_sharpe.append('#F18F01')

    ax3.barh(sharpe_sorted['Model'], sharpe_sorted['Sharpe'], color=colors_sharpe)
    ax3.set_xlabel('Sharpe Ratio (higher is better)')
    ax3.set_title('Sharpe Ratio by Model')
    ax3.invert_yaxis()
    ax3.axvline(x=0, color='red', linestyle='--', alpha=0.5)

    ax4 = axes[1, 1]
    type_metrics = price_summary_df.groupby('Type').agg({
        'RMSE': 'mean',
        'Dir. Acc. (%)': 'mean',
        'Sharpe': 'mean'
    }).round(4)

    x = np.arange(len(type_metrics))
    width = 0.25

    rmse_norm = type_metrics['RMSE'] / type_metrics['RMSE'].max()
    da_norm = type_metrics['Dir. Acc. (%)'] / 100
    sharpe_norm = (type_metrics['Sharpe'] - type_metrics['Sharpe'].min()) / \
                  (type_metrics['Sharpe'].max() - type_metrics['Sharpe'].min() + 1e-6)

    ax4.bar(x - width, rmse_norm, width, label='RMSE (norm)', color='#2E86AB')
    ax4.bar(x, da_norm, width, label='Dir. Acc. (norm)', color='#A23B72')
    ax4.bar(x + width, sharpe_norm, width, label='Sharpe (norm)', color='#F18F01')
    ax4.set_xticks(x)
    ax4.set_xticklabels(type_metrics.index)
    ax4.set_ylabel('Normalized Score')
    ax4.set_title('Average Metrics by Model Type')
    ax4.legend()

    plt.tight_layout()
    plt.savefig(results_dir / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Metrics comparison saved to {results_dir / "metrics_comparison.png"}')
else:
    print('No price models to compare')

In [ ]:
#@title 21. Plot Metrics Comparison - All Volatility Models
if volatility_results:
    vol_metrics_rows = []

    for model_key, result in volatility_results.items():
        m = result['metrics']
        vol_metrics_rows.append({
            'Model': model_key,
            'Best Val QLIKE': m.get('best_val_qlike', np.nan),
            'Best Val R2': m.get('best_val_r2', np.nan),
            'Final Val Loss': m.get('final_val_loss', np.nan),
            'Type': 'Physics-Informed' if ('pinn' in model_key or 'heston' in model_key) else 'Baseline',
        })

    vol_metrics_df = pd.DataFrame(vol_metrics_rows).sort_values('Best Val QLIKE', ascending=True)

    colors = ['#A23B72' if t == 'Physics-Informed' else '#2E86AB' for t in vol_metrics_df['Type']]

    fig, axes = plt.subplots(1, 3, figsize=(18, max(5, 0.45 * len(vol_metrics_df) + 2)))

    axes[0].barh(vol_metrics_df['Model'], vol_metrics_df['Best Val QLIKE'], color=colors)
    axes[0].set_title('Best Validation QLIKE (lower better)')
    axes[0].set_xlabel('QLIKE')
    axes[0].invert_yaxis()

    axes[1].barh(vol_metrics_df['Model'], vol_metrics_df['Best Val R2'], color=colors)
    axes[1].set_title('Best Validation R² (higher better)')
    axes[1].set_xlabel('R²')
    axes[1].invert_yaxis()
    axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

    axes[2].barh(vol_metrics_df['Model'], vol_metrics_df['Final Val Loss'], color=colors)
    axes[2].set_title('Final Validation Loss (lower better)')
    axes[2].set_xlabel('Loss')
    axes[2].invert_yaxis()

    plt.tight_layout()
    plt.savefig(results_dir / 'volatility_metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Volatility metrics comparison saved to {results_dir / "volatility_metrics_comparison.png"}')
else:
    print('No volatility models trained')

In [ ]:
#@title 22. Plot Predictions vs Actuals (All Price Models)
if price_results:
    n_models = len(price_results)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.8 * n_rows))
    axes = np.array(axes).reshape(-1)

    for idx, (model_key, result) in enumerate(price_results.items()):
        ax = axes[idx]

        preds = np.asarray(result['predictions']).flatten()
        targs = np.asarray(result['targets']).flatten()
        n = min(200, len(preds), len(targs))

        ax.plot(targs[:n], label='Actual', alpha=0.8, linewidth=1.6)
        ax.plot(preds[:n], label='Predicted', alpha=0.8, linewidth=1.2)

        rmse = result['metrics'].get('rmse', np.nan)
        ax.set_title(f'{model_key} | RMSE={rmse:.4f}', fontsize=10)
        ax.set_xlabel('Sample')
        ax.set_ylabel('Close (normalized)')
        ax.grid(True, alpha=0.25)

        if idx == 0:
            ax.legend(fontsize=8)

    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Predictions vs Actuals - All Price Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'predictions_vs_actual_all_price_models.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Prediction comparison saved to {results_dir / "predictions_vs_actual_all_price_models.png"}')
else:
    print('No price models trained')

## 10. Visualization Suite (13 figures)
Generates research-grade plots covering loss decomposition, prediction accuracy, regime error, surfaces, equity/drawdown, Sharpe distribution, Monte Carlo fan chart, ablation, and volatility accuracy.

In [ ]:

#@title 10. Unified Visualization Suite (13 research-grade figures)
import json, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

fig_dir = Path(results_dir) / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# Helper utilities

def _to_array(x):
    if x is None:
        return None
    try:
        return np.asarray(x, dtype=float)
    except Exception:
        return None

def _get_series(best_key=None):
    if best_key is None and 'price_summary_df' in globals() and len(price_summary_df) > 0:
        best_key = price_summary_df.loc[price_summary_df['RMSE'].idxmin()]['Model']
    for mk in [best_key] if best_key else []:
        res = price_results.get(mk, {}) if 'price_results' in globals() else {}
        preds = _to_array(res.get('predictions') or res.get('y_pred') or res.get('y_hat'))
        targs = _to_array(res.get('targets') or res.get('y_true'))
        if preds is not None and targs is not None and len(preds) == len(targs):
            return mk, preds, targs
    if 'price_results' in globals():
        for mk, res in price_results.items():
            preds = _to_array(res.get('predictions') or res.get('y_pred') or res.get('y_hat'))
            targs = _to_array(res.get('targets') or res.get('y_true'))
            if preds is not None and targs is not None and len(preds) == len(targs):
                return mk, preds, targs
    return None, None, None

def _rolling_vol(series, window=20):
    s = pd.Series(series)
    return s.rolling(window=window, min_periods=1).std().bfill().values

def _savefig(name):
    plt.tight_layout()
    path = fig_dir / name
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"[saved] {path}")

# 1) Loss decomposition
if 'price_results' in globals() and len(price_results) > 0:
    for mk, res in price_results.items():
        hist = res.get('history', {})
        data = hist.get('train_data_loss') or hist.get('data_loss')
        phys = hist.get('train_physics_loss') or hist.get('physics_loss')
        total = hist.get('train_loss') or None
        if data is not None and phys is not None:
            steps = range(1, len(data)+1)
            plt.figure(figsize=(8,4))
            plt.plot(steps, data, label='Data loss')
            plt.plot(steps, phys, label='Physics loss')
            if total is not None:
                plt.plot(steps, total, label='Total loss', linestyle='--')
            plt.yscale('log')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title(f'Loss Decomposition - {mk}')
            plt.legend()
            _savefig(f"loss_decomposition_{mk}.png")
            break

# Series for downstream plots
model_key, preds, targs = _get_series()
if preds is None or targs is None:
    print("[warn] No predictions/targets found; skipping time-series visualisations")
else:
    time_axis = np.arange(len(preds))
    errors = preds - targs
    regimes = _rolling_vol(targs, window=20)
    try:
        regime_bins = pd.qcut(regimes, q=3, labels=['low','mid','high'])
    except Exception:
        regime_bins = pd.cut(regimes, bins=3, labels=['low','mid','high'], include_lowest=True)

    # 2) Prediction vs actual
    plt.figure(figsize=(10,4))
    plt.plot(time_axis, targs, label='Actual', linewidth=1.5)
    plt.plot(time_axis, preds, label=f'Predicted ({model_key})', linewidth=1.2)
    plt.xlabel('Time')
    plt.ylabel('Standardised close')
    plt.title('Prediction vs Actual (out-of-sample)')
    plt.legend()
    _savefig('prediction_vs_actual.png')

    # 3) Forecast error vs horizon
    horizons = list(range(1, min(30, max(2, len(preds)//3))))
    horizon_err = []
    for h in horizons:
        err = preds[h:] - targs[:-h]
        horizon_err.append(np.sqrt(np.mean(err**2)))
    plt.figure(figsize=(6,4))
    plt.plot(horizons, horizon_err, marker='o')
    plt.xlabel('Forecast horizon (days)')
    plt.ylabel('RMSE')
    plt.title('Forecast Error vs Horizon')
    _savefig('forecast_error_vs_horizon.png')

    # 4) Error heatmap
    df_heat = pd.DataFrame({
        'time': time_axis[:len(regime_bins)],
        'regime': regime_bins.astype(str),
        'error': np.abs(errors[:len(regime_bins)])
    })
    pivot = df_heat.pivot_table(index='regime', columns='time', values='error', aggfunc='mean')
    plt.figure(figsize=(10,3))
    sns.heatmap(pivot, cmap='magma', cbar_kws={'label':'|error|'})
    plt.title('Error Heatmap by Volatility Regime')
    plt.xlabel('Time')
    _savefig('error_heatmap_regime.png')

    # 5) Prediction surface
    grid_time, grid_price = np.meshgrid(np.linspace(0, len(preds)-1, 64), np.linspace(np.min(targs), np.max(targs), 64))
    pred_surface = np.interp(grid_time, time_axis, preds)
    plt.figure(figsize=(6,5))
    cs = plt.contourf(grid_time, grid_price, pred_surface, levels=30, cmap='viridis')
    plt.colorbar(cs, label='Predicted')
    plt.xlabel('Time')
    plt.ylabel('Standardised close')
    plt.title('Prediction Surface (time × price)')
    _savefig('prediction_surface.png')

    # 6) Error surface
    err_surface = np.abs(pred_surface - grid_price)
    plt.figure(figsize=(6,5))
    cs = plt.contourf(grid_time, grid_price, err_surface, levels=30, cmap='inferno')
    plt.colorbar(cs, label='|error|')
    plt.xlabel('Time')
    plt.ylabel('Standardised close')
    plt.title('Prediction Error Surface')
    _savefig('prediction_error_surface.png')

    # 7) Equity curve
    returns = np.diff(targs, prepend=targs[0])
    positions = np.sign(preds)
    strategy_ret = positions * returns
    equity = (1 + strategy_ret).cumprod()
    bh_equity = (1 + returns).cumprod()
    plt.figure(figsize=(10,4))
    plt.plot(equity, label='PINN strategy')
    plt.plot(bh_equity, label='Buy & Hold', linestyle='--')
    plt.xlabel('Time')
    plt.ylabel('Equity (relative)')
    plt.title('Strategy Equity Curve (signal = sign(pred))')
    plt.legend()
    _savefig('equity_curve.png')

    # 8) Drawdown curve
    roll_max = np.maximum.accumulate(equity)
    drawdown = equity / roll_max - 1
    plt.figure(figsize=(10,3))
    plt.fill_between(range(len(drawdown)), drawdown, 0, color='red', alpha=0.3)
    plt.xlabel('Time')
    plt.ylabel('Drawdown')
    plt.title('Drawdown Curve')
    _savefig('drawdown_curve.png')

    # 9) Sharpe distribution (bootstrap)
    sharpe_samples = []
    for _ in range(500):
        sample = np.random.choice(strategy_ret, size=len(strategy_ret), replace=True)
        mu = sample.mean()
        sd = sample.std() + 1e-8
        sharpe_samples.append(mu / sd * math.sqrt(252))
    plt.figure(figsize=(6,4))
    sns.histplot(sharpe_samples, bins=40, kde=True)
    plt.xlabel('Sharpe ratio (bootstrapped)')
    plt.title('Sharpe Ratio Distribution')
    _savefig('sharpe_distribution.png')

    # 10) Ablation (RMSE & Sharpe)
    if 'price_summary_df' in globals() and len(price_summary_df) > 0:
        cmp = price_summary_df[['Model','RMSE','Sharpe']].copy()
        plt.figure(figsize=(8,4))
        sns.barplot(data=cmp, x='Model', y='RMSE')
        plt.xticks(rotation=45, ha='right')
        plt.title('Ablation: RMSE by model')
        _savefig('ablation_rmse.png')
        plt.figure(figsize=(8,4))
        sns.barplot(data=cmp, x='Model', y='Sharpe')
        plt.xticks(rotation=45, ha='right')
        plt.title('Ablation: Sharpe by model')
        _savefig('ablation_sharpe.png')

    # 11) Training stability (variance over epochs)
    selected_res = price_results.get(model_key, {}) if 'price_results' in globals() else {}
    selected_hist = selected_res.get('history', {}) if isinstance(selected_res, dict) else {}
    if isinstance(selected_hist, dict):
        pred_var = selected_hist.get('prediction_variance')
        if pred_var is not None:
            plt.figure(figsize=(6,3))
            plt.plot(pred_var)
            plt.xlabel('Epoch')
            plt.ylabel('Var(pred)')
            plt.title('Training Stability (variance over epochs)')
            _savefig('training_stability.png')

    # 12) Monte Carlo fan chart (residual bootstrap)
    residuals = errors
    horizon = 60
    sims = 200
    paths = []
    for _ in range(sims):
        shock = np.random.choice(residuals, size=horizon, replace=True)
        path = [targs[-1]]
        for s in shock:
            path.append(path[-1] + s)
        paths.append(path[1:])
    paths = np.array(paths)
    mean_path = paths.mean(axis=0)
    lower = np.percentile(paths, 5, axis=0)
    upper = np.percentile(paths, 95, axis=0)
    plt.figure(figsize=(8,4))
    plt.plot(mean_path, label='Mean path', color='blue')
    plt.fill_between(range(horizon), lower, upper, color='blue', alpha=0.2, label='90% band')
    plt.title('Monte Carlo Fan Chart (residual bootstrap)')
    plt.xlabel('Steps ahead')
    plt.ylabel('Standardised close')
    plt.legend()
    _savefig('monte_carlo_fanchart.png')

    # 13) Volatility forecast accuracy (if available)
    if 'volatility_results' in globals() and len(volatility_results) > 0:
        for vk, vres in volatility_results.items():
            vpred = _to_array(vres.get('predictions'))
            vtrue = _to_array(vres.get('targets'))
            if vpred is None or vtrue is None:
                continue
            plt.figure(figsize=(10,3))
            plt.plot(vtrue, label='Realised vol')
            plt.plot(vpred, label='Predicted vol')
            plt.title(f'Volatility Forecast - {vk}')
            plt.xlabel('Time')
            plt.ylabel('Volatility')
            plt.legend()
            _savefig(f'volatility_forecast_{vk}.png')
            break


In [ ]:
#@title 23. Final Comprehensive Report
print('='*80)
print('FINAL TRAINING REPORT - ALL MODELS')
print('='*80)
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Device: {device}')
print(f'Research Config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')

print(f'\n=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}/{len(BASELINE_MODELS)}')
print(f'  - PINN: {len(pinn_results)}/{len(PINN_MODELS)}')
print(f'  - Advanced PINN (incl. Spectral): {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)}')
print(f'  - Financial DP-PINN: {len(financial_dp_results)}/{len(FINANCIAL_DP_MODELS)}')
print(f'  - Volatility: {len(volatility_results)}/{len(VOLATILITY_MODELS)}')
print(f'  - DP PINN: {len(dp_results)}/{len(DP_MODELS)}')

print('\n' + '-'*40)
print('BEST PRICE PREDICTION MODELS')
print('-'*40)

if len(price_summary_df) > 0:
    best_rmse = price_summary_df.loc[price_summary_df['RMSE'].idxmin()]
    print(f'Best RMSE: {best_rmse["Model"]} ({best_rmse["RMSE"]:.4f})')

    best_da = price_summary_df.loc[price_summary_df['Dir. Acc. (%)'].idxmax()]
    print(f'Best Directional Accuracy: {best_da["Model"]} ({best_da["Dir. Acc. (%)"]:.2f}%)')

    best_sharpe = price_summary_df.loc[price_summary_df['Sharpe'].idxmax()]
    print(f'Best Sharpe Ratio: {best_sharpe["Model"]} ({best_sharpe["Sharpe"]:.3f})')

print('\n' + '-'*40)
print('BEST VOLATILITY MODELS')
print('-'*40)

if len(vol_summary_df) > 0:
    best_vol = vol_summary_df.loc[vol_summary_df['Best Val QLIKE'].idxmin()]
    print(f'Best Val QLIKE: {best_vol["Model"]} ({best_vol["Best Val QLIKE"]:.6f})')

print('\n' + '-'*40)
print('PINN vs BASELINE COMPARISON (Price Models)')
print('-'*40)

if len(price_summary_df) > 0:
    baseline_avg = price_summary_df[price_summary_df['Type'] == 'Baseline'].mean(numeric_only=True)
    pinn_avg = price_summary_df[price_summary_df['Type'] == 'PINN'].mean(numeric_only=True)

    if not baseline_avg.empty and not pinn_avg.empty:
        print(f'Average RMSE:')
        print(f'  Baseline: {baseline_avg["RMSE"]:.4f}')
        print(f'  PINN: {pinn_avg["RMSE"]:.4f}')
        improvement = (1 - pinn_avg["RMSE"]/baseline_avg["RMSE"])*100
        print(f'  Improvement: {improvement:.2f}%')

        print(f'\nAverage Directional Accuracy:')
        print(f'  Baseline: {baseline_avg["Dir. Acc. (%)"]:.2f}%')
        print(f'  PINN: {pinn_avg["Dir. Acc. (%)"]:.2f}%')

        print(f'\nAverage Sharpe Ratio:')
        print(f'  Baseline: {baseline_avg["Sharpe"]:.3f}')
        print(f'  PINN: {pinn_avg["Sharpe"]:.3f}')



STRICT_OVERNIGHT_CHECK = True
expected_models = set(PRICE_MODELS + VOLATILITY_MODELS + DP_MODELS)
trained_models = set(all_results.keys())
missing_models = sorted(expected_models - trained_models)

print('\n' + '-'*40)
print('COMPLETENESS CHECK')
print('-'*40)
print(f'Expected models: {len(expected_models)}')
print(f'Trained models: {len(trained_models)}')

if missing_models:
    print(f'Missing models ({len(missing_models)}): {missing_models}')
    if STRICT_OVERNIGHT_CHECK:
        raise RuntimeError(f'Overnight run incomplete. Missing models: {missing_models}')
else:
    print('All expected models were trained successfully.')

print('\n' + '='*80)
print(f'Results saved to: {results_dir}')
print('\nIndividual model result files:')
for model_key in all_results.keys():
    print(f'  - {model_key}_results.json')
print('\nSummary files:')
print('  - price_model_summary.csv')
print('  - volatility_model_summary.csv')
print('  - dp_pinn_summary.csv')
print('\nVisualization files:')
print('  - price_training_curves.png')
print('  - volatility_training_curves.png')
print('  - dp_pinn_training_curves.png')
print('  - metrics_comparison.png')
print('  - volatility_metrics_comparison.png')
print('  - predictions_vs_actual_all_price_models.png')
print('='*80)